# IoT-NetGuard — Model 4: Convolutional Neural Network (CNN)
**Project:** Anomaly Detection for IoT Cybersecurity  
**Dataset:** `iot23_combined.csv` (output of `01_Data_Preprocessing.ipynb`)  
**Algorithm:** Deep Neural Network (Dense layers) with Dropout regularisation  
**Framework:** TensorFlow 2.4 / Keras

## 1. Imports

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')
print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Load Dataset

In [ ]:
FILEPATH = "iot23_combined.csv"
FEATURES = [
    'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes',
    'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
    'proto_icmp', 'proto_tcp', 'proto_udp',
    'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTO', 'conn_state_RSTOS0',
    'conn_state_RSTR', 'conn_state_RSTRH', 'conn_state_S0', 'conn_state_S1',
    'conn_state_S2', 'conn_state_S3', 'conn_state_SF', 'conn_state_SH', 'conn_state_SHR'
]
TARGET = 'label'

df = pd.read_csv(FILEPATH)
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

print(f'Dataset shape : {df.shape}')
print(f'Label distribution:\n{df[TARGET].value_counts().to_string()}')

## 3. Feature Preparation & Normalisation

In [ ]:
X = df[FEATURES].values

# One-hot encode labels (required for categorical_crossentropy)
Y_ohe  = pd.get_dummies(df[TARGET]).values
CLASSES = pd.get_dummies(df[TARGET]).columns.tolist()
N_CLASSES = Y_ohe.shape[1]

# Normalise features and labels
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_Y = MinMaxScaler()
Y_scaled = scaler_Y.fit_transform(Y_ohe)

X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled, Y_scaled, test_size=0.2, random_state=10
)

print(f'Classes    : {CLASSES}')
print(f'N classes  : {N_CLASSES}')
print(f'X_train    : {X_train.shape}')
print(f'X_test     : {X_test.shape}')

## 4. Build Model Architecture

In [ ]:
model = Sequential([
    Dense(2000, activation='relu', input_dim=len(FEATURES)),
    Dense(1500, activation='relu'),
    Dropout(0.2),
    Dense(800,  activation='relu'),
    Dropout(0.2),
    Dense(400,  activation='relu'),
    Dropout(0.2),
    Dense(150,  activation='relu'),
    Dropout(0.2),
    Dense(N_CLASSES, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

## 5. Train the Model

In [ ]:
EPOCHS     = 10
BATCH_SIZE = 256

print(f'Training for {EPOCHS} epochs, batch size {BATCH_SIZE}...')
start = time.time()

history = model.fit(
    X_train, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, Y_test),
    verbose=1
)

elapsed = time.time() - start
print(f'\nTraining complete in {elapsed:.2f} seconds ({elapsed/60:.1f} min)')

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Model Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Model Loss', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('IoT-NetGuard CNN — Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/cnn_training_history.png', dpi=150)
plt.show()
print('Saved → outputs/cnn_training_history.png')

## 7. Evaluation

In [ ]:
# Decode one-hot predictions back to class labels
Y_pred_proba = model.predict(X_test)
Y_pred_idx   = np.argmax(Y_pred_proba, axis=1)
Y_true_idx   = np.argmax(Y_test, axis=1)

Y_pred_labels = [CLASSES[i] for i in Y_pred_idx]
Y_true_labels = [CLASSES[i] for i in Y_true_idx]

accuracy = accuracy_score(Y_true_labels, Y_pred_labels)

print(f'Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'Time cost : {elapsed:.2f} seconds')
print()
print('Classification Report:')
print(classification_report(Y_true_labels, Y_pred_labels))

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(Y_true_labels, Y_pred_labels, labels=CLASSES)

plt.figure(figsize=(13, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('CNN — Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('outputs/cnn_confusion_matrix.png', dpi=150)
plt.show()
print('Saved → outputs/cnn_confusion_matrix.png')

## 9. Summary

In [ ]:
final_val_acc = history.history['val_accuracy'][-1]
print('=' * 52)
print('  IoT-NetGuard — CNN Summary')
print('=' * 52)
print(f'  Model       : Sequential Dense CNN')
print(f'  Layers      : 5 Dense + 4 Dropout')
print(f'  Features    : {len(FEATURES)}')
print(f'  Classes     : {N_CLASSES}')
print(f'  Epochs      : {EPOCHS}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Val Acc     : {final_val_acc*100:.2f}%')
print(f'  Test Acc    : {accuracy*100:.2f}%')
print(f'  Time        : {elapsed:.2f}s')
print('=' * 52)